In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
import nltk
from collections import Counter

nltk.download('punkt')

# =========================
# 1. LOAD DATA
# =========================

dataset = load_dataset("squad")
data = dataset["train"]

pairs = [(item["question"], item["answers"]["text"][0]) for item in data]
pairs = pairs[:5000]

print("Total pairs:", len(pairs))

# =========================
# 2. TOKENIZER
# =========================

def tokenize(text):
    return nltk.word_tokenize(text.lower())

# =========================
# 3. VOCAB
# =========================

counter = Counter()
for q, a in pairs:
    counter.update(tokenize(q))
    counter.update(tokenize(a))

PAD, UNK, SOS, EOS = "<PAD>", "<UNK>", "<SOS>", "<EOS>"

vocab_size = 15000
most_common = counter.most_common(vocab_size - 4)

idx2word = [PAD, UNK, SOS, EOS] + [w for w, _ in most_common]
word2idx = {w: i for i, w in enumerate(idx2word)}

print("Vocab:", len(word2idx))

# =========================
# 4. ENCODE
# =========================

MAX_LEN = 30

def encode(text):
    tokens = [SOS] + tokenize(text) + [EOS]
    ids = [word2idx.get(t, word2idx[UNK]) for t in tokens]

    if len(ids) < MAX_LEN:
        ids += [word2idx[PAD]] * (MAX_LEN - len(ids))
    else:
        ids = ids[:MAX_LEN]

    return ids

# =========================
# 5. DATASET
# =========================

class QADataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        q, a = self.pairs[idx]
        return torch.tensor(encode(q)), torch.tensor(encode(a))

dataset = QADataset(pairs)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

# =========================
# Positional Encoding
# =========================

class PositionalEncoding(nn.Module):
    def __init__(self, embed_size, max_len=100):
        super().__init__()

        encoding = torch.zeros(max_len, embed_size)
        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, embed_size, 2) * (-torch.log(torch.tensor(10000.0)) / embed_size)
        )

        encoding[:, 0::2] = torch.sin(position * div_term)
        encoding[:, 1::2] = torch.cos(position * div_term)

        self.encoding = encoding.unsqueeze(0)

    def forward(self, x):
        return x + self.encoding[:, :x.size(1), :]

# =========================
# MultiHead Attention
# =========================

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_size, heads=8):
        super().__init__()
        self.embed_size = embed_size
        self.heads = heads
        self.head_dim = embed_size // heads

        self.values = nn.Linear(embed_size, embed_size)
        self.keys = nn.Linear(embed_size, embed_size)
        self.queries = nn.Linear(embed_size, embed_size)

        self.fc_out = nn.Linear(embed_size, embed_size)

    def forward(self, query, key, value):
        N, query_len, _ = query.shape
        _, key_len, _ = key.shape

        V = self.values(value)
        K = self.keys(key)
        Q = self.queries(query)

        V = V.view(N, key_len, self.heads, self.head_dim)
        K = K.view(N, key_len, self.heads, self.head_dim)
        Q = Q.view(N, query_len, self.heads, self.head_dim)

        energy = torch.einsum("nqhd,nkhd->nhqk", Q, K)
        attention = torch.softmax(energy / (self.head_dim ** 0.5), dim=3)

        out = torch.einsum("nhql,nlhd->nqhd", attention, V)
        out = out.reshape(N, query_len, self.embed_size)

        return self.fc_out(out)

# =========================
# Encoder
# =========================

class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.pos = PositionalEncoding(embed_size)
        self.attn = MultiHeadAttention(embed_size)
        self.norm = nn.LayerNorm(embed_size)

    def forward(self, x):
        x = self.embedding(x)
        x = self.pos(x)

        attn = self.attn(x, x, x)
        x = self.norm(x + attn)
        return x

# =========================
# Decoder
# =========================

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.pos = PositionalEncoding(embed_size)

        self.self_attn = MultiHeadAttention(embed_size)
        self.cross_attn = MultiHeadAttention(embed_size)

        self.norm1 = nn.LayerNorm(embed_size)
        self.norm2 = nn.LayerNorm(embed_size)
        self.norm3 = nn.LayerNorm(embed_size)

        self.ff = nn.Sequential(
            nn.Linear(embed_size, embed_size * 4),
            nn.ReLU(),
            nn.Linear(embed_size * 4, embed_size)
        )

        self.fc = nn.Linear(embed_size, vocab_size)

    def forward(self, x, enc_out):
        x = self.embedding(x)
        x = self.pos(x)

        attn1 = self.self_attn(x, x, x)
        x = self.norm1(x + attn1)

        attn2 = self.cross_attn(x, enc_out, enc_out)
        x = self.norm2(x + attn2)

        ff = self.ff(x)
        x = self.norm3(x + ff)

        return self.fc(x)

# =========================
# Seq2Seq
# =========================

class Seq2Seq(nn.Module):
    def __init__(self, vocab_size, embed_size):
        super().__init__()
        self.encoder = Encoder(vocab_size, embed_size)
        self.decoder = Decoder(vocab_size, embed_size)

    def forward(self, src, trg):
        enc_out = self.encoder(src)
        return self.decoder(trg, enc_out)

model = Seq2Seq(len(word2idx), 128)

# =========================
# TRAINING
# =========================

criterion = nn.CrossEntropyLoss(ignore_index=word2idx[PAD])
optimizer = optim.Adam(model.parameters(), lr=0.0005)

for epoch in range(10):
    total_loss = 0

    for questions, answers in loader:
        optimizer.zero_grad()

        outputs = model(questions, answers[:, :-1])
        target = answers[:, 1:]

        outputs = outputs.reshape(-1, outputs.shape[-1])
        target = target.reshape(-1)

        loss = criterion(outputs, target)
        loss.backward()

        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.2f}")

# =========================
# SIMPLE PREDICT
# =========================

def predict_sentence(text):
    model.eval()
    src = torch.tensor([encode(text)])
    trg = torch.tensor([[word2idx[SOS]] + [word2idx[PAD]]*(MAX_LEN-1)])

    with torch.no_grad():
        output = model(src, trg)
        predicted = torch.argmax(output[0], dim=1)

    result = []
    for idx in predicted:
        word = idx2word[idx.item()]
        if word in [PAD, SOS]:
            continue
        if word == EOS:
            break
        result.append(word)

    return " ".join(result)

# =========================
# TEST
# =========================

print("\nOld Model:\n")
print(predict_sentence("Who is he?"))
print(predict_sentence("What is Python?"))

# =========================
# T5 MODEL
# =========================

from transformers import T5Tokenizer, T5ForConditionalGeneration

tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-small")
model_t5 = T5ForConditionalGeneration.from_pretrained("google/flan-t5-small")

input_text = "question: Who is Chopin?"
input_ids = tokenizer(input_text, return_tensors="pt").input_ids

outputs = model_t5.generate(input_ids, max_length=50)

print("\nT5 Model:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Total pairs: 5000
Vocab: 8198
Epoch 1, Loss: 1000.28
Epoch 2, Loss: 821.01
Epoch 3, Loss: 720.27
Epoch 4, Loss: 618.06
Epoch 5, Loss: 509.36
Epoch 6, Loss: 397.92
Epoch 7, Loss: 295.84
Epoch 8, Loss: 206.79
Epoch 9, Loss: 135.46
Epoch 10, Loss: 81.17

Old Model:





Loading weights: 100%|█████████████████████████████████████████████████████████████| 190/190 [00:00<00:00, 1565.74it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



T5 Model:

a sailor
